In [18]:

# PROJECT SETUP

from pathlib import Path
import os
import sys
import time

import pandas as pd
from IPython.display import display


PROJECT_ROOT = Path(
    r"C:\Users\SCHOOL\Desktop\FinMark_DataPipeline"
)

SRC_DIRECTORY = PROJECT_ROOT / "src"

os.chdir(PROJECT_ROOT)

for path in [PROJECT_ROOT, SRC_DIRECTORY]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

pd.set_option(
    "display.max_columns",
    None,
)

pd.set_option(
    "display.width",
    200,
)

print("=" * 90)
print("FINMARK DATA ANALYTICS PIPELINE")
print("=" * 90)
print("Project Folder :", PROJECT_ROOT)
print("Working Folder :", os.getcwd())
print("Python         :", sys.executable)
print("=" * 90)

FINMARK DATA ANALYTICS PIPELINE
Project Folder : C:\Users\SCHOOL\Desktop\FinMark_DataPipeline
Working Folder : C:\Users\SCHOOL\Desktop\FinMark_DataPipeline
Python         : C:\Users\SCHOOL\anaconda3\python.exe


In [19]:

# HELPER FUNCTIONS

def stage_header(
    stage_number,
    process_name,
):
    print("\n" + "=" * 90)
    print(
        f"STAGE {stage_number} – "
        f"{process_name}"
    )
    print("=" * 90)


def stage_footer(
    start_time,
    summary,
):
    duration = (
        time.perf_counter()
        - start_time
    )

    print("\nPROCESS SUMMARY")
    print("-" * 90)

    for label, value in summary.items():
        print(
            f"{label:<28}: {value}"
        )

    print("-" * 90)

    print(
        f"{'Process Duration':<28}: "
        f"{duration:.4f} seconds"
    )

    print(
        f"{'Status':<28}: COMPLETED"
    )

    print("=" * 90)

    return duration


def count_rows(
    datasets,
):
    return sum(
        len(dataframe)
        for dataframe
        in datasets.values()
    )


def count_columns(
    datasets,
):
    return sum(
        len(dataframe.columns)
        for dataframe
        in datasets.values()
    )


def count_missing_values(
    datasets,
):
    return sum(
        int(
            dataframe
            .isna()
            .sum()
            .sum()
        )
        for dataframe
        in datasets.values()
    )


def count_duplicate_rows(
    datasets,
):
    return sum(
        int(
            dataframe
            .duplicated()
            .sum()
        )
        for dataframe
        in datasets.values()
    )

In [20]:

# STAGE 1: DATA INGESTION

from config import (
    create_project_directories,
)
from ingestion import load_datasets


stage_header(
    1,
    "DATA INGESTION",
)

stage1_start = time.perf_counter()

create_project_directories()

(
    datasets,
    ingestion_metadata,
) = load_datasets()

display(
    ingestion_metadata[
        [
            "dataset",
            "row_count",
            "column_count",
            "file_size_bytes",
            "status",
        ]
    ]
)

stage1_summary = {
    "Datasets Loaded": len(datasets),
    "Total Rows Loaded": (
        f"{count_rows(datasets):,}"
    ),
    "Total Columns Loaded": (
        f"{count_columns(datasets):,}"
    ),
    "Successful Loads": int(
        ingestion_metadata[
            "status"
        ]
        .astype(str)
        .str.upper()
        .eq("LOADED")
        .sum()
    ),
}

stage1_duration = stage_footer(
    stage1_start,
    stage1_summary,
)

2026-07-12 02:39:59,082 | INFO | finmark_pipeline.ingestion | Stage started: Data Ingestion
2026-07-12 02:39:59,083 | INFO | finmark_pipeline.ingestion | Loading dataset | Name: event_logs | Path: C:\Users\SCHOOL\Desktop\FinMark_DataPipeline\data\raw\event_logs.csv
2026-07-12 02:39:59,095 | INFO | finmark_pipeline.ingestion | Dataset loaded successfully | Name: event_logs | Rows: 2000 | Columns: 50
2026-07-12 02:39:59,096 | INFO | finmark_pipeline.ingestion | Loading dataset | Name: marketing_summary | Path: C:\Users\SCHOOL\Desktop\FinMark_DataPipeline\data\raw\marketing_summary.csv
2026-07-12 02:39:59,100 | INFO | finmark_pipeline.ingestion | Dataset loaded successfully | Name: marketing_summary | Rows: 100 | Columns: 50
2026-07-12 02:39:59,101 | INFO | finmark_pipeline.ingestion | Loading dataset | Name: trend_report | Path: C:\Users\SCHOOL\Desktop\FinMark_DataPipeline\data\raw\trend_report.csv
2026-07-12 02:39:59,105 | INFO | finmark_pipeline.ingestion | Dataset loaded successfully 


STAGE 1 – DATA INGESTION


,dataset,row_count,column_count,file_size_bytes,status
0,event_logs,2000,50,358145,LOADED
1,marketing_summary,100,50,24810,LOADED
2,trend_report,20,50,5648,LOADED



PROCESS SUMMARY
------------------------------------------------------------------------------------------
Datasets Loaded             : 3
Total Rows Loaded           : 2,120
Total Columns Loaded        : 150
Successful Loads            : 3
------------------------------------------------------------------------------------------
Process Duration            : 0.0320 seconds
Status                      : COMPLETED


In [21]:

# VALIDATION AND PROFILING

from validation import (
    validate_datasets,
)
from profiling import (
    profile_datasets,
)


stage_header(
    2,
    "DATA VALIDATION & PROFILING",
)

stage2_start = time.perf_counter()

(
    validation_results,
    validation_report,
    validation_mode,
) = validate_datasets(
    datasets
)

quality_report = profile_datasets(
    datasets
)

display(
    validation_report
)

display(
    quality_report
)

warning_count = 0
error_count = 0

if (
    "warning_count"
    in validation_report.columns
):
    warning_count = int(
        pd.to_numeric(
            validation_report[
                "warning_count"
            ],
            errors="coerce",
        )
        .fillna(0)
        .sum()
    )

if (
    "error_count"
    in validation_report.columns
):
    error_count = int(
        pd.to_numeric(
            validation_report[
                "error_count"
            ],
            errors="coerce",
        )
        .fillna(0)
        .sum()
    )

stage2_summary = {
    "Datasets Validated": len(datasets),
    "Validation Records": len(
        validation_report
    ),
    "Quality Records": len(
        quality_report
    ),
    "Warning Count": warning_count,
    "Error Count": error_count,
    "Pipeline Mode": validation_mode,
}

stage2_duration = stage_footer(
    stage2_start,
    stage2_summary,
)


2026-07-12 02:40:25 | INFO | finmark_pipeline | Logger initialized.
2026-07-12 02:40:25 | INFO | finmark_pipeline | Log directory: C:\Users\SCHOOL\Desktop\FinMark_DataPipeline\src\logs
2026-07-12 02:40:25 | INFO | finmark_pipeline | Schema validation started for dataset: event_logs
2026-07-12 02:40:25 | WARNING | finmark_pipeline | event_logs: 45 unexpected columns detected.
2026-07-12 02:40:25 | INFO | finmark_pipeline | Schema validation completed for event_logs | Mode: NORMAL | Warnings: 1 | Errors: 0
2026-07-12 02:40:25 | INFO | finmark_pipeline | Schema validation started for dataset: marketing_summary
2026-07-12 02:40:25 | WARNING | finmark_pipeline | marketing_summary: 45 unexpected columns detected.
2026-07-12 02:40:25 | INFO | finmark_pipeline | Schema validation completed for marketing_summary | Mode: NORMAL | Warnings: 1 | Errors: 0
2026-07-12 02:40:25 | INFO | finmark_pipeline | Schema validation started for dataset: trend_report
2026-07-12 02:40:25 | WARNING | finmark_pipe


STAGE 2 – DATA VALIDATION & PROFILING

SCHEMA VALIDATION SUMMARY
----------------------------------------------------------------------------------------------------
          dataset pipeline_mode missing_required_columns  unexpected_columns_count  warning_count  error_count
       event_logs        NORMAL                                                 45              1            0
marketing_summary        NORMAL                                                 45              1            0
     trend_report        NORMAL                                                 47              1            0
----------------------------------------------------------------------------------------------------
Overall Pipeline Mode: NORMAL

DATA QUALITY SUMMARY
-----------------------------------------------------------------------------------------------
          dataset  row_count  column_count  missing_count  completeness_score duplicate_rows quality_status
       event_logs       2000    

,dataset,is_valid,pipeline_mode,missing_required_columns,missing_critical_columns,missing_optional_columns,unexpected_columns_count,unexpected_columns,corrupted_numeric_values,invalid_datetime_values,warning_count,error_count,warnings,errors
0,event_logs,True,NORMAL,,,,45,"col_10, col_11, col_12, col_13, col_14, col_15...",{'amount': 0},{'event_time': 0},1,0,event_logs: 45 unexpected columns detected.,
1,marketing_summary,True,NORMAL,,,,45,"col_10, col_11, col_12, col_13, col_14, col_15...","{'users_active': 0, 'total_sales': 0, 'new_cus...","{'date': 0, 'report_generated': 0}",1,0,marketing_summary: 45 unexpected columns detec...,
2,trend_report,True,NORMAL,,,,47,"col_10, col_11, col_12, col_13, col_14, col_15...","{'avg_users': 0, 'sales_growth_rate': 0}",{},1,0,trend_report: 47 unexpected columns detected.,


,record_type,dataset,column,row_count,column_count,missing_count,non_missing_count,missing_percentage,completeness_score,unique_count,duplicate_rows,duplicate_percentage,is_critical,quality_status,recommended_action
0,DATASET,event_logs,,2000,50,26671,73329,26.67,73.33,,0,0.0,,WARNING,CONTINUE
1,COLUMN,event_logs,user_id,2000,50,0,2000,0.00,100.00,400,,,True,GOOD,RETAIN
2,COLUMN,event_logs,event_type,2000,50,0,2000,0.00,100.00,8,,,True,GOOD,RETAIN
3,COLUMN,event_logs,event_time,2000,50,0,2000,0.00,100.00,1759,,,True,GOOD,RETAIN
4,COLUMN,event_logs,product_id,2000,50,0,2000,0.00,100.00,30,,,False,GOOD,RETAIN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148,COLUMN,trend_report,col_46,20,50,3,17,15.00,85.00,17,,,False,WARNING,RETAIN
149,COLUMN,trend_report,col_47,20,50,4,16,20.00,80.00,16,,,False,WARNING,RETAIN
150,COLUMN,trend_report,col_48,20,50,5,15,25.00,75.00,3,,,False,WARNING,RETAIN
151,COLUMN,trend_report,col_49,20,50,8,12,40.00,60.00,12,,,False,WARNING,RETAIN



PROCESS SUMMARY
------------------------------------------------------------------------------------------
Datasets Validated          : 3
Validation Records          : 3
Quality Records             : 153
Warning Count               : 3
Error Count                 : 0
Pipeline Mode               : NORMAL
------------------------------------------------------------------------------------------
Process Duration            : 0.0834 seconds
Status                      : COMPLETED


In [22]:

# STAGE 3: FALLBACK, CLEANING AND QUANRANTINE

from fallback import (
    apply_fallback_rules,
)
from cleaning import (
    clean_datasets,
)


stage_header(
    3,
    "FALLBACK, CLEANING & QUARANTINE",
)

stage3_start = time.perf_counter()

(
    processed_datasets,
    quarantined_datasets,
    fallback_result,
    fallback_action_report,
) = apply_fallback_rules(
    datasets
)

(
    cleaned_datasets,
    cleaning_report,
) = clean_datasets(
    processed_datasets
)

display(
    fallback_action_report
)

display(
    cleaning_report
)

total_quarantined = sum(
    len(dataframe)
    for dataframe
    in quarantined_datasets.values()
)

stage3_summary = {
    "Processed Datasets": len(
        processed_datasets
    ),
    "Cleaned Datasets": len(
        cleaned_datasets
    ),
    "Rows After Cleaning": (
        f"{count_rows(cleaned_datasets):,}"
    ),
    "Fallback Actions": len(
        fallback_action_report
    ),
    "Quarantined Rows": (
        f"{total_quarantined:,}"
    ),
    "Fallback Mode": (
        fallback_result.pipeline_mode
    ),
    "Financial KPIs": (
        "ENABLED"
        if fallback_result
        .financial_kpis_enabled
        else "DISABLED"
    ),
}

stage3_duration = stage_footer(
    stage3_start,
    stage3_summary,
)

2026-07-12 02:41:02 | WARNING | finmark_pipeline | The missing transaction amount rate exceeded the safe threshold. Affected records were quarantined and financial KPIs were disabled.
2026-07-12 02:41:02 | INFO | finmark_pipeline | Fallback action report saved to: C:\Users\SCHOOL\Desktop\FinMark_DataPipeline\src\output\fallback_actions.csv
2026-07-12 02:41:02 | INFO | finmark_pipeline | event_logs: Removing 45 undocumented columns.
2026-07-12 02:41:02 | INFO | finmark_pipeline | Cleaning completed for event_logs | Rows: 1858 -> 1858 | Columns: 52 -> 7 | Duplicates removed: 0 | Remaining missing values: 874
2026-07-12 02:41:02 | INFO | finmark_pipeline | marketing_summary: Removing 45 undocumented columns.
2026-07-12 02:41:02 | INFO | finmark_pipeline | Cleaning completed for marketing_summary | Rows: 100 -> 100 | Columns: 50 -> 5 | Duplicates removed: 0 | Remaining missing values: 0
2026-07-12 02:41:02 | INFO | finmark_pipeline | trend_report: Removing 47 undocumented columns.
2026-07-


STAGE 3 – FALLBACK, CLEANING & QUARANTINE

BUSINESS-RULE FALLBACK SUMMARY
------------------------------------------------------------------------
Pipeline Mode: DEGRADED
Financial KPIs Enabled: False
Valid Behavioral Nulls: 874
Imputed Transaction Amounts: 0
Quarantined Transaction Rows: 142
Unknown Event Nulls: 0
------------------------------------------------------------------------

CLEANING SUMMARY
----------------------------------------------------------------------------------------------------
          dataset  rows_before  rows_after  duplicates_removed  columns_before  columns_after  remaining_missing_values
       event_logs         1858        1858                   0              52              7                       874
marketing_summary          100         100                   0              50              5                         0
     trend_report           20          20                   0              50              3                         0
----------

,dataset,column,issue,action,affected_rows,details
0,event_logs,amount,missing_behavioral_amount,retained_as_valid_null,874,Behavioral activities do not necessarily invol...
1,event_logs,amount,critical_missing_transaction_amount,quarantined_and_disabled_financial_kpis,142,Missing rate: 54.62%


,dataset,rows_before,rows_after,rows_removed,duplicates_removed,columns_before,columns_after,columns_removed,remaining_missing_values,status
0,event_logs,1858,1858,0,0,52,7,45,874,CLEANED
1,marketing_summary,100,100,0,0,50,5,45,0,CLEANED
2,trend_report,20,20,0,0,50,3,47,0,CLEANED



PROCESS SUMMARY
------------------------------------------------------------------------------------------
Processed Datasets          : 3
Cleaned Datasets            : 3
Rows After Cleaning         : 1,978
Fallback Actions            : 2
Quarantined Rows            : 142
Fallback Mode               : DEGRADED
Financial KPIs              : DISABLED
------------------------------------------------------------------------------------------
Process Duration            : 0.1110 seconds
Status                      : COMPLETED


In [23]:

# STAGE 4: TRANSFORMATION AND STORAGE

from transformation import (
    transform_datasets,
)
from storage import (
    save_all_zones,
)


stage_header(
    4,
    "DATA TRANSFORMATION & STORAGE",
)

stage4_start = time.perf_counter()

(
    transformed_datasets,
    curated_datasets,
) = transform_datasets(
    cleaned_datasets
)

storage_report = save_all_zones(
    transformed_datasets=
        transformed_datasets,
    curated_datasets=
        curated_datasets,
    quarantined_datasets=
        quarantined_datasets,
)

display(
    storage_report
)

stage4_summary = {
    "Transformed Datasets": len(
        transformed_datasets
    ),
    "Transformed Rows": (
        f"{count_rows(transformed_datasets):,}"
    ),
    "Curated Datasets": len(
        curated_datasets
    ),
    "Curated Rows": (
        f"{count_rows(curated_datasets):,}"
    ),
    "Storage Records": len(
        storage_report
    ),
}

stage4_duration = stage_footer(
    stage4_start,
    stage4_summary,
)

2026-07-12 02:41:35 | INFO | finmark_pipeline | Event Logs transformation completed | Rows: 1858 | Invalid dates: 0
2026-07-12 02:41:35 | INFO | finmark_pipeline | Marketing Summary transformation completed | Rows: 100 | Invalid dates: 0
2026-07-12 02:41:35 | INFO | finmark_pipeline | Trend Report transformation completed | Rows: 20 | Invalid weeks: 0
2026-07-12 02:41:35 | INFO | finmark_pipeline | Dataset saved: C:\Users\SCHOOL\Desktop\FinMark_DataPipeline\src\data\clean\event_logs_clean.csv | Rows: 1858 | Columns: 15 | Size: 252018 bytes
2026-07-12 02:41:35 | INFO | finmark_pipeline | Dataset saved: C:\Users\SCHOOL\Desktop\FinMark_DataPipeline\src\data\clean\marketing_summary_clean.csv | Rows: 100 | Columns: 9 | Size: 10438 bytes
2026-07-12 02:41:35 | INFO | finmark_pipeline | Dataset saved: C:\Users\SCHOOL\Desktop\FinMark_DataPipeline\src\data\clean\trend_report_clean.csv | Rows: 20 | Columns: 6 | Size: 941 bytes
2026-07-12 02:41:35 | INFO | finmark_pipeline | Dataset saved: C:\User


STAGE 4 – DATA TRANSFORMATION & STORAGE

TRANSFORMATION SUMMARY
--------------------------------------------------------------------------------
event_logs                   1858 rows       15 columns
marketing_summary             100 rows        9 columns
trend_report                   20 rows        6 columns

CURATED DATASETS
--------------------------------------------------------------------------------
event_classification_report              15 rows
daily_activity_summary                    6 rows
event_type_summary                        8 rows
daily_sales_summary                     100 rows
monthly_sales_summary                     4 rows
--------------------------------------------------------------------------------

STORAGE SUMMARY
---------------------------------------------------------------------------------------------------------
storage_zone                     dataset  row_count  column_count  file_size_bytes status
       CLEAN                  event_logs       1

,file_name,file_path,row_count,column_count,file_size_bytes,status,storage_zone,dataset
0,event_logs_clean.csv,C:\Users\SCHOOL\Desktop\FinMark_DataPipeline\s...,1858,15,252018,SAVED,CLEAN,event_logs
1,marketing_summary_clean.csv,C:\Users\SCHOOL\Desktop\FinMark_DataPipeline\s...,100,9,10438,SAVED,CLEAN,marketing_summary
2,trend_report_clean.csv,C:\Users\SCHOOL\Desktop\FinMark_DataPipeline\s...,20,6,941,SAVED,CLEAN,trend_report
3,event_classification_report.csv,C:\Users\SCHOOL\Desktop\FinMark_DataPipeline\s...,15,4,597,SAVED,CURATED,event_classification_report
4,daily_activity_summary.csv,C:\Users\SCHOOL\Desktop\FinMark_DataPipeline\s...,6,6,278,SAVED,CURATED,daily_activity_summary
5,event_type_summary.csv,C:\Users\SCHOOL\Desktop\FinMark_DataPipeline\s...,8,7,622,SAVED,CURATED,event_type_summary
6,daily_sales_summary.csv,C:\Users\SCHOOL\Desktop\FinMark_DataPipeline\s...,100,5,4587,SAVED,CURATED,daily_sales_summary
7,monthly_sales_summary.csv,C:\Users\SCHOOL\Desktop\FinMark_DataPipeline\s...,4,6,324,SAVED,CURATED,monthly_sales_summary
8,event_logs_quarantine.csv,C:\Users\SCHOOL\Desktop\FinMark_DataPipeline\s...,142,53,36055,SAVED,QUARANTINE,event_logs



PROCESS SUMMARY
------------------------------------------------------------------------------------------
Transformed Datasets        : 3
Transformed Rows            : 1,978
Curated Datasets            : 5
Curated Rows                : 133
Storage Records             : 9
------------------------------------------------------------------------------------------
Process Duration            : 0.0903 seconds
Status                      : COMPLETED


In [24]:

# STAGE 5: BUSINESS ANALYTICS

from src.business_analytics import (
    BusinessAnalytics,
)


stage_header(
    5,
    "BUSINESS ANALYTICS LAYER",
)

stage5_start = time.perf_counter()


def select_dataset(
    dataset_name,
):
    if (
        dataset_name
        in curated_datasets
    ):
        return curated_datasets[
            dataset_name
        ]

    if (
        dataset_name
        in transformed_datasets
    ):
        return transformed_datasets[
            dataset_name
        ]

    if (
        dataset_name
        in cleaned_datasets
    ):
        return cleaned_datasets[
            dataset_name
        ]

    return datasets.get(
        dataset_name,
        pd.DataFrame(),
    )


def build_dataset_summary(
    dataframe,
):
    row_count = len(dataframe)
    column_count = len(
        dataframe.columns
    )

    total_cells = (
        row_count
        * column_count
    )

    missing_count = int(
        dataframe
        .isna()
        .sum()
        .sum()
    )

    duplicate_rows = int(
        dataframe
        .duplicated()
        .sum()
    )

    completeness_score = (
        (
            total_cells
            - missing_count
        )
        / total_cells
        * 100
        if total_cells > 0
        else 0.0
    )

    duplicate_rate = (
        duplicate_rows
        / row_count
        * 100
        if row_count > 0
        else 0.0
    )

    quality_score = max(
        0.0,
        completeness_score
        - duplicate_rate,
    )

    return {
        "row_count": row_count,
        "column_count": column_count,
        "missing_count": missing_count,
        "duplicate_rows": duplicate_rows,
        "completeness_score": round(
            completeness_score,
            2,
        ),
        "quality_score": round(
            quality_score,
            2,
        ),
    }


event_logs_analytics = (
    select_dataset(
        "event_logs"
    )
)

marketing_analytics = (
    select_dataset(
        "marketing_summary"
    )
)

trend_analytics = (
    select_dataset(
        "trend_report"
    )
)

dataset_summaries = {
    "event_logs":
        build_dataset_summary(
            event_logs_analytics
        ),
    "marketing_summary":
        build_dataset_summary(
            marketing_analytics
        ),
    "trend_report":
        build_dataset_summary(
            trend_analytics
        ),
}

analytics = BusinessAnalytics(
    output_directory=(
        PROJECT_ROOT / "output"
    )
)

stage5_results = analytics.run(
    event_logs=
        event_logs_analytics,
    marketing_summary=
        marketing_analytics,
    trend_report=
        trend_analytics,
    dataset_summaries=
        dataset_summaries,
    pipeline_mode=
        fallback_result.pipeline_mode,
    financial_kpis_enabled=
        fallback_result
        .financial_kpis_enabled,
    quarantined_records=
        total_quarantined,
    warning_count=(
        1
        if total_quarantined > 0
        else 0
    ),
    error_count=0,
    total_processing_seconds=0.0,
)

display(
    stage5_results[
        "business_kpis"
    ][
        [
            "category",
            "metric",
            "value",
            "unit",
            "status",
        ]
    ]
)

display(
    stage5_results[
        "pipeline_health_metrics"
    ][
        [
            "metric",
            "value",
            "unit",
        ]
    ]
)

business_kpis = stage5_results[
    "business_kpis"
]

pipeline_health_metrics = (
    stage5_results[
        "pipeline_health_metrics"
    ]
)

executive_summary = (
    stage5_results[
        "executive_summary"
    ]
)

powerbi_dataset = (
    stage5_results[
        "powerbi_dashboard_dataset"
    ]
)

stage5_summary = {
    "Business KPI Records": len(
        business_kpis
    ),
    "Health Metric Records": len(
        pipeline_health_metrics
    ),
    "Executive Summary Rows": len(
        executive_summary
    ),
    "Dashboard Dataset Rows": len(
        powerbi_dataset
    ),
    "Dashboard Dataset Columns": len(
        powerbi_dataset.columns
    ),
}

stage5_duration = stage_footer(
    stage5_start,
    stage5_summary,
)


STAGE 5 – BUSINESS ANALYTICS LAYER


,category,metric,value,unit,status
0,Customer Activity,Processed Events,1858.00,records,AVAILABLE
1,Customer Activity,Unique Users,399.00,users,AVAILABLE
2,Customer Activity,Unique Event Types,8.00,event types,AVAILABLE
3,Customer Activity,Active Event Days,6.00,days,AVAILABLE
4,Customer Activity,Average Events Per Day,309.67,events/day,AVAILABLE
5,Customer Activity,Average Events Per User,4.66,events/user,AVAILABLE
6,Financial Performance,Event Log Transaction Value,0.00,currency,DISABLED
7,Financial Performance,Average Event Transaction Value,0.00,currency,DISABLED
8,Financial Performance,Event Transaction Count,0.00,transactions,DISABLED
9,Sales Performance,Total Sales,5767580.45,currency,AVAILABLE


,metric,value,unit
0,Pipeline Health Score,91.27,score/100
1,Average Completeness Score,98.95,percent
2,Average Data Quality Score,98.95,percent
3,Total Processed Rows,1978,rows
4,Total Missing Values,874,values
5,Duplicate Rows,0,rows
6,Pipeline Warning Count,1,warnings
7,Pipeline Error Count,0,errors
8,Pipeline Mode,DEGRADED,status
9,Total Processing Time,0.0,seconds



PROCESS SUMMARY
------------------------------------------------------------------------------------------
Business KPI Records        : 22
Health Metric Records       : 10
Executive Summary Rows      : 1
Dashboard Dataset Rows      : 100
Dashboard Dataset Columns   : 14
------------------------------------------------------------------------------------------
Process Duration            : 0.0378 seconds
Status                      : COMPLETED


In [25]:

# STAGE 6: DASHBOARD MONITORING

from src.dashboard_monitoring import (
    DashboardMonitoring,
)


stage_header(
    6,
    "DASHBOARD MONITORING & PIPELINE HEALTH",
)

stage6_start = time.perf_counter()

monitoring = DashboardMonitoring(
    output_directory=(
        PROJECT_ROOT / "output"
    )
)

stage6_results = monitoring.run(
    business_kpis=
        business_kpis,
    pipeline_health_metrics=
        pipeline_health_metrics,
    fallback_summary={
        "pipeline_mode":
            fallback_result
            .pipeline_mode,
        "financial_kpis_enabled":
            fallback_result
            .financial_kpis_enabled,
    },
)

dashboard_status = (
    stage6_results[
        "dashboard_status"
    ]
)

dashboard_alerts = (
    stage6_results[
        "dashboard_alerts"
    ]
)

display(
    dashboard_status
)

display(
    dashboard_alerts[
        [
            "alert_category",
            "severity",
            "status",
            "message",
        ]
    ]
)

warning_alerts = int(
    dashboard_alerts[
        "severity"
    ]
    .astype(str)
    .str.upper()
    .eq("WARNING")
    .sum()
)

critical_alerts = int(
    dashboard_alerts[
        "severity"
    ]
    .astype(str)
    .str.upper()
    .eq("CRITICAL")
    .sum()
)

current_dashboard_status = (
    dashboard_status.iloc[0][
        "dashboard_status"
    ]
    if not dashboard_status.empty
    else "UNKNOWN"
)

stage6_summary = {
    "Dashboard Status Rows": len(
        dashboard_status
    ),
    "Total Alerts": len(
        dashboard_alerts
    ),
    "Warning Alerts": warning_alerts,
    "Critical Alerts": critical_alerts,
    "Current Status": (
        current_dashboard_status
    ),
}

stage6_duration = stage_footer(
    stage6_start,
    stage6_summary,
)


STAGE 6 – DASHBOARD MONITORING & PIPELINE HEALTH


,dashboard_status,pipeline_health_score,total_sales,average_daily_sales,average_active_users,total_new_customers,warning_count,critical_count,last_updated
0,WARNING,91.27,5767580.45,57675.8,273.31,765,3,0,2026-07-12T02:42:37.483500


,alert_category,severity,status,message
0,Pipeline Health,INFO,HEALTHY,Pipeline health score is 91.27. The overall pi...
1,Data Quality,WARNING,REVIEW REQUIRED,142 records were quarantined and excluded from...
2,Financial Analytics,WARNING,PARTIALLY DISABLED,Event-log financial KPIs are disabled because ...
3,Pipeline Mode,WARNING,DEGRADED,The pipeline completed using fallback controls...



PROCESS SUMMARY
------------------------------------------------------------------------------------------
Dashboard Status Rows       : 1
Total Alerts                : 4
Warning Alerts              : 3
Critical Alerts             : 0
Current Status              : WARNING
------------------------------------------------------------------------------------------
Process Duration            : 0.0174 seconds
Status                      : COMPLETED


In [26]:

# STAGE 7: DASHBOARD DATA VALIDATION

from dashboard.data_loader import (
    load_dashboard_data,
)


stage_header(
    7,
    "EXECUTIVE DASHBOARD PREPARATION",
)

stage7_start = time.perf_counter()

dashboard_data = (
    load_dashboard_data()
)

stage7_inventory = pd.DataFrame(
    [
        {
            "dataset": name,
            "rows": len(dataframe),
            "columns": len(
                dataframe.columns
            ),
            "status": (
                "READY"
                if not dataframe.empty
                else "EMPTY"
            ),
        }
        for name, dataframe
        in dashboard_data.items()
    ]
)

display(
    stage7_inventory
)

ready_count = int(
    stage7_inventory[
        "status"
    ]
    .eq("READY")
    .sum()
)

empty_count = int(
    stage7_inventory[
        "status"
    ]
    .eq("EMPTY")
    .sum()
)

total_dashboard_rows = int(
    stage7_inventory[
        "rows"
    ].sum()
)

stage7_summary = {
    "Datasets Checked": len(
        stage7_inventory
    ),
    "Ready Datasets": ready_count,
    "Empty Datasets": empty_count,
    "Total Dashboard Rows": (
        f"{total_dashboard_rows:,}"
    ),
    "Dashboard Status": (
        "READY"
        if empty_count == 0
        else "INCOMPLETE"
    ),
}

stage7_duration = stage_footer(
    stage7_start,
    stage7_summary,
)


STAGE 7 – EXECUTIVE DASHBOARD PREPARATION


,dataset,rows,columns,status
0,event_logs,2000,50,READY
1,marketing_summary,100,50,READY
2,trend_report,20,50,READY
3,business_kpis,22,7,READY
4,pipeline_health,10,7,READY
5,executive_summary,1,12,READY
6,dashboard_alerts,4,4,READY
7,dashboard_status,1,9,READY
8,powerbi_dataset,100,14,READY



PROCESS SUMMARY
------------------------------------------------------------------------------------------
Datasets Checked            : 9
Ready Datasets              : 9
Empty Datasets              : 0
Total Dashboard Rows        : 2,258
Dashboard Status            : READY
------------------------------------------------------------------------------------------
Process Duration            : 0.0422 seconds
Status                      : COMPLETED


In [27]:

# FINAL DATA PIPELINE SUMMARY

pipeline_summary = pd.DataFrame(
    [
        {
            "Stage": 1,
            "Process":
                "Data Ingestion",
            "Duration Seconds":
                stage1_duration,
            "Status": "COMPLETED",
        },
        {
            "Stage": 2,
            "Process":
                "Validation & Profiling",
            "Duration Seconds":
                stage2_duration,
            "Status": "COMPLETED",
        },
        {
            "Stage": 3,
            "Process":
                "Fallback, Cleaning & Quarantine",
            "Duration Seconds":
                stage3_duration,
            "Status": "COMPLETED",
        },
        {
            "Stage": 4,
            "Process":
                "Transformation & Storage",
            "Duration Seconds":
                stage4_duration,
            "Status": "COMPLETED",
        },
        {
            "Stage": 5,
            "Process":
                "Business Analytics",
            "Duration Seconds":
                stage5_duration,
            "Status": "COMPLETED",
        },
        {
            "Stage": 6,
            "Process":
                "Dashboard Monitoring",
            "Duration Seconds":
                stage6_duration,
            "Status": "COMPLETED",
        },
        {
            "Stage": 7,
            "Process":
                "Dashboard Preparation",
            "Duration Seconds":
                stage7_duration,
            "Status": "COMPLETED",
        },
    ]
)

pipeline_summary[
    "Duration Seconds"
] = pipeline_summary[
    "Duration Seconds"
].round(4)

display(
    pipeline_summary
)

total_pipeline_duration = float(
    pipeline_summary[
        "Duration Seconds"
    ].sum()
)

print("\n" + "=" * 90)
print("FINAL PIPELINE SUMMARY")
print("=" * 90)

print(
    f"{'Stages Completed':<28}: "
    f"{len(pipeline_summary)}"
)

print(
    f"{'Successful Stages':<28}: "
    f"{pipeline_summary['Status'].eq('COMPLETED').sum()}"
)

print(
    f"{'Total Pipeline Duration':<28}: "
    f"{total_pipeline_duration:.4f} seconds"
)

print(
    f"{'Final Status':<28}: "
    f"PIPELINE COMPLETED SUCCESSFULLY"
)

print("=" * 90)


,Stage,Process,Duration Seconds,Status
0,1,Data Ingestion,0.0320,COMPLETED
1,2,Validation & Profiling,0.0834,COMPLETED
2,3,"Fallback, Cleaning & Quarantine",0.1110,COMPLETED
3,4,Transformation & Storage,0.0903,COMPLETED
4,5,Business Analytics,0.0378,COMPLETED
5,6,Dashboard Monitoring,0.0174,COMPLETED
6,7,Dashboard Preparation,0.0422,COMPLETED



FINAL PIPELINE SUMMARY
Stages Completed            : 7
Successful Stages           : 7
Total Pipeline Duration     : 0.4141 seconds
Final Status                : PIPELINE COMPLETED SUCCESSFULLY
